#### Procedure Parameters (WASM)

In [ ]:
def runwasmtime(wasmfile):
  from wasmtime import Engine, Store, Module, Linker, Func, FuncType, ValType

  # P0lib implementation in Python
  def write(i: int):
    print(i)

  def writeln():
    print()

  def read() -> int:
    return int(input())

  # Create engine, store, and module
  engine = Engine()
  store = Store(engine)
  module = Module(store.engine, open(wasmfile, 'rb').read())
  # Use Linker to create the P0lib library
  write_func = Func(store, FuncType([ValType.i32()], []), write)
  writeln_func = Func(store, FuncType([], []), writeln)
  read_func = Func(store, FuncType([], [ValType.i32()]), read)
  linker = Linker(engine)
  linker.define(store, 'P0lib', 'write', write_func)
  linker.define(store, 'P0lib', 'writeln', writeln_func)
  linker.define(store, 'P0lib', 'read', read_func)
  # Create and run instance
  instance = linker.instantiate(store, module)

Consider the following program in a hypothetical extension of P0. Procedure `map` applies procedure parameter `f` to each element of array parameter `a`:

```Pascal
const N = 3
type A = [0..N - 1] → integer
type F = (x: integer) → (y: integer)
procedure map(f: F, a: A) → (b: A)
    var i: integer
        i := 0
        while i < N do
            b[i] ← f(a[i]); i := i + 1
procedure square(x: integer) → (y: integer)
    y := x ⨯ x
program mapsquare
    var a, b: A
        a[0] := 3; a[1] := 1; a[2] := 4
        b ← map(square, a)
        write(b[2])
```

Translate this by hand to WebAssembly so that `call_indirect` makes the call to `f` in `map`. You can use the generated P0 code of fragments of the above code. Add comments to all places in your code that deal with indirect calls.

In [ ]:
%%writefile mapsquare.wat
(module
(import "P0lib" "write" (func $write (param i32)))
(import "P0lib" "writeln" (func $writeln))
(import "P0lib" "read" (func $read (result i32)))

;; type signature for procedure parameter F: (x: integer) -> (y: integer)
;; used by call_indirect to verify the function signature at runtime
(type $F (func (param i32) (result i32)))

;; function table for indirect calls: holds references to callable functions
(table 1 funcref)
;; populate table: square is placed at index 0
(elem (i32.const 0) $square)

(global $_memsize (mut i32) i32.const 0)

;; procedure square(x: integer) -> (y: integer)
;;   y := x * x
(func $square (param $x i32) (result i32)
(local $y i32)
local.get $x
local.get $x
i32.mul
local.set $y
local.get $y
)

;; procedure map(f: F, a: A) -> (b: A)
;; f is passed as a table index (i32) for indirect calling
;; a is passed as a memory address (i32)
;; returns memory address of result array b
(func $map (param $f i32) (param $a i32) (result i32)
(local $b i32)
(local $i i32)
(local $0 i32)
;; save _memsize for deallocation at procedure exit
global.get $_memsize
;; allocate result array b: N * 4 = 12 bytes
global.get $_memsize
local.tee $b
i32.const 12
i32.add
global.set $_memsize
;; i := 0
i32.const 0
local.set $i
;; while i < N do
loop
local.get $i
i32.const 3
i32.lt_s
if
;; compute destination address b[i] = b + i * 4
local.get $b
local.get $i
i32.const 4
i32.mul
i32.add
;; compute a[i]: load from memory at a + i * 4
local.get $a
local.get $i
i32.const 4
i32.mul
i32.add
i32.load
;; indirect call: f(a[i]) — push table index, then call_indirect
;; this calls the function at table[$f] with signature $F
local.get $f
call_indirect (type $F)
;; store the result of f(a[i]) into b[i]
i32.store
;; i := i + 1
local.get $i
i32.const 1
i32.add
local.set $i
br 1
end
end
;; restore _memsize (deallocate local array b)
global.set $_memsize
;; return address of b (caller will copy the data)
local.get $b
)

;; program mapsquare
(func $program
(local $a i32)
(local $b i32)
(local $0 i32)
;; save _memsize for deallocation at program exit
global.get $_memsize
;; allocate arrays a and b: each N * 4 = 12 bytes
global.get $_memsize
local.tee $a
i32.const 12
i32.add
local.tee $b
i32.const 12
i32.add
global.set $_memsize
;; a[0] := 3
local.get $a
i32.const 3
i32.store
;; a[1] := 1
local.get $a
i32.const 4
i32.add
i32.const 1
i32.store
;; a[2] := 4
local.get $a
i32.const 8
i32.add
i32.const 4
i32.store
;; b <- map(square, a)
;; push destination address (b) for memory.copy after the call
local.get $b
;; indirect call setup: pass table index 0 for square
;; the table index is how procedure parameters are passed for indirect calls
i32.const 0
local.get $a
call $map
;; map returns address of result array; copy 12 bytes to b
i32.const 12
memory.copy
;; write(b[2])
local.get $b
i32.const 8
i32.add
i32.load
call $write
;; restore _memsize
global.set $_memsize
)
(memory 1)
(start $program)
)

In [ ]:
!wat2wasm mapsquare.wat

Running the program should print `16`:

In [ ]:
runwasmtime('mapsquare.wasm')